WARNING: this example is not yet fully functional

In [1]:
from line_solver import *
GlobalConstants.setVerbose(VerboseLevel.STD)

A basic M/M/1 with explicit definition of a classwitch node
Recommended ClassSwitch declaration style

In [2]:
model = Network('mm1cs')

# Block 1: nodes
node = np.empty(4, dtype=object)
node[0] = Source(model, 'Source 1')
node[1] = Queue(model, 'Queue 1', SchedStrategy.FCFS)
node[2] = Sink(model, 'Sink 1')
node[3] = ClassSwitch(model, 'ClassSwitch 1')

# Block 2: classes
jobclass = np.empty(2, dtype=object)
jobclass[0] = OpenClass(model, 'Class1', 0)
jobclass[1] = OpenClass(model, 'Class2', 0)

node[0].setArrival(jobclass[0], Exp.fitMean(10.00)) # (Source 1,Class1)
node[0].setArrival(jobclass[1], Exp.fitMean(2.00)) # (Source 1,Class2)
node[1].setService(jobclass[0], Exp.fitMean(1.00)) # (Queue 1,Class1)
node[1].setService(jobclass[1], Exp.fitMean(1.00)) # (Queue 1,Class2)

# Block 3: topology
# The class switching matrix can now be declared after the classes, so the
# ClassSwitch node can be declared outside Block 1.
csmatrix = node[3].initClassSwitchMatrix() # element (i,j) = probability that class i switches to j
csmatrix[0][0] = 0.3
csmatrix[0][1] = 0.7
csmatrix[1][0] = 1.0
node[3].setClassSwitchingMatrix(csmatrix)

P = model.initRoutingMatrix() # initialize routing matrix
P.set(jobclass[0],jobclass[0],node[0],node[3],1.0) # (Source 1,Class1) -> (ClassSwitch 1,Class1)
P.set(jobclass[0],jobclass[0],node[1],node[2],1.0) # (Queue 1,Class1) -> (Sink 1,Class1)
P.set(jobclass[0],jobclass[0],node[3],node[1],1.0) # (ClassSwitch 1,Class1) -> (Queue 1,Class1)
P.set(jobclass[1],jobclass[1],node[0],node[3],1.0) # (Source 1,Class2) -> (ClassSwitch 1,Class2)
P.set(jobclass[1],jobclass[1],node[1],node[2],1.0) # (Queue 1,Class2) -> (Sink 1,Class2)
P.set(jobclass[1],jobclass[1],node[3],node[1],1.0) # (ClassSwitch 1,Class2) -> (Queue 1,Class2)
model.link(P);

In [3]:
# TODO: getAvgChainTable not yet available in JLINE
AvgTable = SolverJMT(model).getAvgTable()
#AvgTable = SolverJMT(model).getAvgChainTable()
#print(AvgTable)

JMT Model: /tmp/workspace/jsim/12570488628685118818/model.jsim
JMT Analysis completed. Runtime: 1.719 seconds
    Station JobClass      QLen      Util     RespT    ResidT      ArvR  \
0  Source 1   Class1  0.000000  0.000000  0.000000  0.000000  0.101417   
1  Source 1   Class2  0.000000  0.000000  0.000000  0.000000  0.495042   
2   Queue 1   Class1  1.429134  0.528605  2.534820  2.239091  0.527209   
3   Queue 1   Class2  0.169740  0.068087  2.427392  0.283196  0.071193   

       Tput  
0  0.101417  
1  0.495042  
2  0.527209  
3  0.071193  
